# Assignment 8 — Person 2: embeddings, retrieval and scaling

This notebook analyzes actual saved CPU retrieval measurements. **Generation and subjective quality reviews are pending the team's shared Person 3 runner.** The 15 matrix rows are retrieval results, not completed RAG evaluations. No generation output is invented.

The shared corpus and three chunkers belong to Person 1. See `person2/README.md` for setup, model sources, interfaces and timing definitions. Model weights are downloaded only when inference mode is enabled.

In [1]:
from pathlib import Path
import sys, json, statistics
import pandas as pd
from IPython.display import display, Markdown
ROOT = Path.cwd()
if not (ROOT / 'a8' / 'corpus.py').exists() and ROOT.name == 'a8':
    ROOT = ROOT.parent
assert (ROOT / 'a8' / 'corpus.py').exists(), 'Start from repository root or a8/'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from a8.corpus import load_chunks, load_paragraphs, load_queries, STRATEGIES, word_count
from a8.retrieval import MODELS, load_encoder, load_index, score_retrieval
from a8.person2_experiments import run_retrieval_experiments
RUN_RETRIEVAL = False
RUN_DIR = ROOT / 'a8/results/person2/small-cpu-2026-11-08'
if RUN_RETRIEVAL:
    # Choose a fresh directory for new hardware/configuration; preserve original evidence.
    RUN_DIR = ROOT / 'a8/results/person2/my-new-run'
    encoder = load_encoder('small', device='cpu')
    run_retrieval_experiments(RUN_DIR, encoder, top_k=3, repetitions=10)
manifest = json.loads((RUN_DIR / 'run_manifest.json').read_text())
rows = [json.loads(line) for line in (RUN_DIR / 'retrieval.jsonl').read_text().splitlines()]
assert len(rows) == 25 and len({r['row_id'] for r in rows}) == 25
assert all(r['config_sha256'] == manifest['config_sha256'] for r in rows)
print('Saved run:', RUN_DIR.relative_to(ROOT))
print('Model:', manifest['model']['model_id'], 'revision:', manifest['model']['revision'])
print('Device:', manifest['model']['device'], 'context limit:', manifest['model']['max_seq_length'])

Saved run: a8/results/person2/small-cpu-2026-11-08
Model: sentence-transformers/all-MiniLM-L6-v2 revision: 1110a243fdf4706b3f48f1d95db1a4f5529b4d41
Device: cpu context limit: 256


## Corpus and selected embedding models

In [2]:
display(pd.DataFrame([dict(size=size, **spec) for size,spec in MODELS.items()]))
summary = []
for dataset in ['small', 'full']:
    paragraphs = load_paragraphs(dataset)
    summary.append(dict(dataset=dataset, paragraphs=len(paragraphs),
                        words=sum(word_count(p.text) for p in paragraphs),
                        **{s: len(load_chunks(s, dataset)) for s in STRATEGIES}))
display(pd.DataFrame(summary))
# Validate answer evidence in both datasets without passing it to any model.
for dataset in ['small','full']:
    paragraphs = {p.id:p.text for p in load_paragraphs(dataset)}
    for q in load_queries():
        text = ' '.join(paragraphs[p] for p in q['supporting_paragraph_ids'])
        assert all(quote in text for quote in q['supporting_quotes'])
for strategy in STRATEGIES:
    full = {c.id:c.text for c in load_chunks(strategy,'full')}
    assert all(full[c.id] == c.text for c in load_chunks(strategy,'small'))
print('Evidence retained across scaling datasets; chunk IDs and text remain stable.')

,size,model_id,dimension,license,max_seq_length,query_prefix
0,small,sentence-transformers/all-MiniLM-L6-v2,384,Apache-2.0,256,
1,medium,sentence-transformers/all-mpnet-base-v2,768,Apache-2.0,384,
2,large,BAAI/bge-large-en-v1.5,1024,MIT,512,Represent this sentence for searching relevant...


,dataset,paragraphs,words,fixed,overlap,hybrid
0,small,15,1461,15,10,10
1,full,30,2901,30,20,20


Evidence retained across scaling datasets; chunk IDs and text remain stable.


## Fifteen matrix retrieval rows

Source recall and reciprocal rank are paragraph-location diagnostics. Quote coverage checks actual text; it is not a subjective quality score. Pending context/answer quality and generation timing fields are intentionally empty.

In [3]:
matrix = pd.DataFrame(r for r in rows if r['experiment']=='matrix')
assert len(matrix)==15 and len(matrix[['strategy','query_id']].drop_duplicates())==15
columns = ['embedding_model','strategy','query_id','query','source_recall_at_k',
           'source_reciprocal_rank','evidence_quote_coverage','retrieved_context_quality',
           'answer_quality','retrieval_ms','generation_ms','end_to_end_ms','notes']
display(matrix[columns])

,embedding_model,strategy,query_id,query,source_recall_at_k,source_reciprocal_rank,evidence_quote_coverage,retrieved_context_quality,answer_quality,retrieval_ms,generation_ms,end_to_end_ms,notes
0,sentence-transformers/all-MiniLM-L6-v2,fixed,Q1,"I finished closing the kitchen, but the dishwa...",1.0,1.0,1.0,None,None,20.564458,None,None,Retrieval measured; shared generator and human...
1,sentence-transformers/all-MiniLM-L6-v2,fixed,Q2,The store ran out of an item on our shared gro...,1.0,1.0,1.0,None,None,30.828542,None,None,Retrieval measured; shared generator and human...
2,sentence-transformers/all-MiniLM-L6-v2,fixed,Q3,I need the shared cordless drill this afternoo...,1.0,1.0,1.0,None,None,30.951500,None,None,Retrieval measured; shared generator and human...
3,sentence-transformers/all-MiniLM-L6-v2,fixed,Q4,The bathroom tap has a slow drip even with the...,1.0,1.0,1.0,None,None,12.046875,None,None,Retrieval measured; shared generator and human...
4,sentence-transformers/all-MiniLM-L6-v2,fixed,Q5,A friend is staying overnight. What needs to b...,1.0,1.0,1.0,None,None,7.995666,None,None,Retrieval measured; shared generator and human...
5,sentence-transformers/all-MiniLM-L6-v2,overlap,Q1,"I finished closing the kitchen, but the dishwa...",1.0,1.0,1.0,None,None,9.210583,None,None,Retrieval measured; shared generator and human...
6,sentence-transformers/all-MiniLM-L6-v2,overlap,Q2,The store ran out of an item on our shared gro...,1.0,1.0,1.0,None,None,6.848291,None,None,Retrieval measured; shared generator and human...
7,sentence-transformers/all-MiniLM-L6-v2,overlap,Q3,I need the shared cordless drill this afternoo...,1.0,1.0,1.0,None,None,6.525292,None,None,Retrieval measured; shared generator and human...
8,sentence-transformers/all-MiniLM-L6-v2,overlap,Q4,The bathroom tap has a slow drip even with the...,1.0,1.0,1.0,None,None,6.497625,None,None,Retrieval measured; shared generator and human...
9,sentence-transformers/all-MiniLM-L6-v2,overlap,Q5,A friend is staying overnight. What needs to b...,1.0,1.0,1.0,None,None,9.315417,None,None,Retrieval measured; shared generator and human...


## Retrieved chunks — complete evidence

In [4]:
for row in rows:
    if row['experiment'] != 'matrix':
        continue
    print('\n' + '='*72)
    print(row['strategy'], row['query_id'], row['query'])
    for hit in row['retrieved_chunks']:
        print(f"\nRank {hit['rank']} | {hit['chunk_id']} | cosine={hit['score']:.4f}")
        print('Paragraphs:', ', '.join(hit['paragraph_ids']))
        print(hit['text'])
    print('Answer: pending shared generator')


fixed Q1 I finished closing the kitchen, but the dishwasher is still running. What should I mark on the board and what should I do with the dishwasher?

Rank 1 | fixed-chores-01 | cosine=0.6557
Paragraphs: chores-01, chores-02
The kitchen closer takes the evening handoff after the last meal. The schedule assigns one closer for each evening and lists a backup for weeks when a resident travels. Before starting, the closer checks the task board for notes about items soaking or food waiting to cool. A note should identify the item and the resident who will finish it. The closer can complete the rest of the routine while that resident handles the exception. Clean cookware belongs in the drying rack until it is dry enough to put away. Keep the main counter available for breakfast preparation. When the evening kitchen close is finished, mark the existing Kitchen Close card complete on the household task board. If the dishwasher is running, clip the Running tag to its handle and leave unloadi

## Controlled scaling comparison

Same small model, hybrid strategy, k=3 and five queries. Compare smaller/full corpus while all reference answers remain present. Timings exclude model load/index building. These are retrieval timings, not end-to-end RAG response times.

In [5]:
scaling = pd.DataFrame(r for r in rows if r['experiment']=='scaling')
assert len(scaling)==10
samples = pd.DataFrame(json.loads((RUN_DIR/'scaling_latency_samples.json').read_text()))
display(scaling[['dataset','query_id','source_recall_at_k','source_reciprocal_rank','evidence_quote_coverage']])
display(samples.groupby('dataset')['retrieval_ms'].agg(['count','median','min','max']))
display(pd.DataFrame(json.loads((RUN_DIR/'index_stats.json').read_text())).T)
display(Markdown((ROOT/'a8/person2/FINDINGS.md').read_text()))

,dataset,query_id,source_recall_at_k,source_reciprocal_rank,evidence_quote_coverage
0,small,Q1,1.0,1.0,1.0
1,small,Q2,1.0,1.0,1.0
2,small,Q3,1.0,1.0,1.0
3,small,Q4,1.0,1.0,1.0
4,small,Q5,1.0,1.0,1.0
5,full,Q1,1.0,1.0,1.0
6,full,Q2,1.0,1.0,1.0
7,full,Q3,1.0,1.0,1.0
8,full,Q4,1.0,1.0,1.0
9,full,Q5,1.0,1.0,1.0


,count,median,min,max
dataset,,,,
full,50,4.839771,3.535833,18.007167
small,50,4.899750,3.636084,14.553709


,build_ms,chunk_count,vector_bytes,max_chunk_tokens,embedding_context_limit,paragraph_count,word_count
full/fixed,2087.752750,30.0,46080.0,167.0,256.0,30.0,2901.0
full/overlap,384.623708,20.0,30720.0,233.0,256.0,30.0,2901.0
full/hybrid,424.659208,20.0,30720.0,233.0,256.0,30.0,2901.0
small/hybrid,223.273750,10.0,15360.0,229.0,256.0,15.0,1461.0


# Person 2 retrieval findings

CPU run on November 8, 2026. Small model: `sentence-transformers/all-MiniLM-L6-v2`, 384 dimensions, k=3. These are retrieval findings; generated answers, shared quality ratings and end-to-end RAG latency are pending.

## Main matrix

All 15 matrix rows achieved source recall@3 = 1.0, source reciprocal rank = 1.0 and exact supporting-quote coverage = 1.0. Each strategy recovered the answer evidence within its top three chunks. This ceiling on five controlled queries does not establish equal performance on harder questions or equal answer quality. Full retrieved text and cosine scores are preserved in the raw JSONL.

## Scaling comparison

Fixed configuration: small model, hybrid chunking, k=3 and the same five queries. All answer evidence remains in the smaller dataset. The larger dataset adds 15 paragraphs of neighboring topics; it is a distractor experiment, not a missing-answer test.

| Dataset | Paragraphs | Words | Chunks | Raw vector bytes | Build ms | Median warmed retrieval ms | Range ms |
|---|---:|---:|---:|---:|---:|---:|---:|
| small | 15 | 1461 | 10 | 15360 | 223.27 | 4.90 | 3.64–14.55 |
| full | 30 | 2901 | 20 | 30720 | 424.66 | 4.84 | 3.54–18.01 |

Each timing distribution has 50 observations (10 repeats × five queries), interleaved between datasets. The small timing difference is run variability, not evidence of a meaningful speed advantage. Vector bytes exclude text, JSON, model weights and compression overhead. Build times are one observation per index, not repeated benchmarks.

Both datasets retain perfect top-three evidence coverage and the supporting source remains at rank one for all five queries. Lower-ranked chunk IDs change for Q1, Q2, Q3 and Q5; Q4 keeps the same three IDs. Added-topic chunks enter the full-dataset results in four queries. This shows competition from neighboring topics without loss of the reference evidence.

Noise cannot be inferred from source IDs alone: related context may be useful, while a chunk from the expected source may add redundant material. A shared-rubric context relevance/noise review is still needed. No observed effect on generated answer quality can be claimed yet.

## Limits and next steps

- Longest small-model chunk: 233 tokens, within the standard 256-token limit. No chunk was silently truncated.
- Five queries and 10 versus 20 hybrid chunks are too small to establish production-scale search performance.
- Query encoding dominates such a tiny index; vector-memory growth is linear here.
- Keep the same generator/prompt/decoding across all team experiments and complete the 15 answer and quality fields with actual outputs.
- Model families differ in training and architecture; cross-model results must not attribute every difference to embedding dimension.


## Verify persisted indexes and model smoke tests

In [6]:
for directory in sorted((RUN_DIR/'indexes').iterdir()):
    expected = json.loads((directory/'manifest.json').read_text())
    index = load_index(directory, expected)
    print(directory.name, index.vectors.shape, 'checksums valid')
smoke_file = ROOT/'a8/results/person2/model_smoke_tests.json'
if smoke_file.exists():
    smoke = json.loads(smoke_file.read_text())
    display(pd.DataFrame([{k:v for k,v in item.items() if k!='hits'} for item in smoke]))

full-fixed (30, 384) checksums valid
full-hybrid (20, 384) checksums valid
full-overlap (20, 384) checksums valid
small-hybrid (10, 384) checksums valid


,size,model_id,dimension,license,max_seq_length,query_prefix,revision,device,normalization,dtype,chunk_count,shape,max_chunk_tokens,query_id,status
0,small,sentence-transformers/all-MiniLM-L6-v2,384,Apache-2.0,256,,1110a243fdf4706b3f48f1d95db1a4f5529b4d41,cpu,l2,float32,20,"[20, 384]",233,Q1,passed
1,medium,sentence-transformers/all-mpnet-base-v2,768,Apache-2.0,384,,e8c3b32edf5434bc2275fc9bab85f82640a19130,cpu,l2,float32,20,"[20, 768]",233,Q1,passed
2,large,BAAI/bge-large-en-v1.5,1024,MIT,512,Represent this sentence for searching relevant...,d4aa6901d3a41ba39fb536a557fa166f842b0e09,cpu,l2,float32,20,"[20, 1024]",233,Q1,passed


## Generation integration — pending

Person 3 supplies the shared generator, prompt and rubric. Its adapter must return the actual answer, exact rendered prompt, measured generation latency and confirmation that context was not truncated. Call `attach_generation(RUN_DIR, generate_answer, generator_manifest)` from `a8.person2_experiments` once these are available.

`generation.jsonl` is written separately so retrieval evidence stays intact. The adapter reports a sum of stage latencies; true end-to-end timing must be measured by the common joint runner. Complete quality ratings only after inspecting actual retrieved contexts and answers using the team rubric. Person 4 integrates this notebook and results into the final submission.